In [7]:
import fitz

temp_pdf_path = r"D:\Nexensus_Projects\IDBI\AOC-4 XBRL_Form AOC4XBRL_15_09_2025.pdf"
doc = fitz.open(temp_pdf_path)

section_text = "CAG details"

page_number = None
coords = None

for i, page in enumerate(doc):
    matches = page.search_for(section_text)  # NO flags

    if matches:
        page_number = i + 1
        rect = matches[0]

        coords = {
            "x0": rect.x0,
            "y0": rect.y0,
            "x1": rect.x1,
            "y1": rect.y1,
            "width": rect.width,
            "height": rect.height,
        }
        break

doc.close()

print("Page number:", page_number)
print("Coordinates:", coords)


Page number: 3
Coordinates: {'x0': 25.56800079345703, 'y0': 234.57997131347656, 'x1': 73.26799011230469, 'y1': 248.45997619628906, 'width': 47.699989318847656, 'height': 13.8800048828125}


In [5]:
########################### Code to open a particular section of a pdf in chrome #########################
import fitz
import subprocess
import os

pdf_path = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf"
section_text = "Part B - Balance Sheet"

doc = fitz.open(pdf_path)

page_number = None
for i, page in enumerate(doc):
    if section_text.lower() in page.get_text().lower():
        page_number = i
        break

if page_number is None:
    raise Exception("Section not found")

# Extract that page
new_doc = fitz.open()
new_doc.insert_pdf(doc, from_page=page_number, to_page=page_number)

output_pdf = f"section_page_{page_number+1}.pdf"
new_doc.save(output_pdf)
new_doc.close()

# Open in Chrome
subprocess.run([
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    os.path.abspath(output_pdf)
])


CompletedProcess(args=['C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe', 'd:\\Nexensus_Projects\\NN\\section_page_6.pdf'], returncode=0)

In [ ]:
########################### Code to open a particular section of a pdf in adobe #########################
import fitz  # PyMuPDF
import subprocess
import os
import urllib.parse


doc = fitz.open(r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf")
section_text = "Part B - Balance Sheet"
safe_text = urllib.parse.quote(section_text)

page_number = None
for i, page in enumerate(doc):
    if section_text.lower() in page.get_text().lower():
        page_number = i + 1
        break

if page_number:
    print(f"Section found on page {page_number}")
else:
    print("Section not found")


pdf_path = os.path.abspath(r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025 - Copy.pdf")

subprocess.Popen([
    r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe",
    f"/A",
    # f"page={page_number}",
    f"page={page_number}",
    pdf_path
])


Section found on page 6


In [ ]:
# ########################### Open a particular section of a PDF in Adobe (Safe Reopen) ###########################

# import fitz  # PyMuPDF
# import subprocess
# import os
# import time
# import win32gui
# import win32process
# import psutil

# PDF_PATH = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025.pdf"
# ADOBE_EXE = r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe"
# SECTION_TEXT = "Part B - Balance Sheet"


# # -------------------- Find page number --------------------
# doc = fitz.open(PDF_PATH)

# page_number = None
# for i, page in enumerate(doc):
#     if SECTION_TEXT.lower() in page.get_text().lower():
#         page_number = i + 1
#         break

# doc.close()

# if not page_number:
#     print("Section not found")
#     exit()

# print(f"Section found on page {page_number}")


# # -------------------- Close PDF if already open in Adobe --------------------
# def close_pdf_if_open(pdf_path):
#     pdf_path = os.path.abspath(pdf_path).lower()

#     def enum_handler(hwnd, _):
#         if win32gui.IsWindowVisible(hwnd):
#             title = win32gui.GetWindowText(hwnd).lower()
#             if pdf_path.split("\\")[-1] in title:
#                 _, pid = win32process.GetWindowThreadProcessId(hwnd)
#                 try:
#                     p = psutil.Process(pid)
#                     if "acrobat" in p.name().lower():
#                         print("Closing already opened PDF in Adobe...")
#                         win32gui.PostMessage(hwnd, 0x0010, 0, 0)  # WM_CLOSE
#                 except psutil.NoSuchProcess:
#                     pass

#     win32gui.EnumWindows(enum_handler, None)


# close_pdf_if_open(PDF_PATH)
# time.sleep(2)  # allow Adobe to close cleanly


# # -------------------- Open PDF at target page & search text --------------------
# subprocess.Popen([
#     ADOBE_EXE,
#     "/A",
#     f"page={page_number}",
#     os.path.abspath(PDF_PATH)
# ])


Section found on page 7


<Popen: returncode: None args: ['C:\\Program Files\\Adobe\\Acrobat DC\\Acrob...>

In [1]:
######################## Open PDF section in Adobe (NO popup) by closing all tabs ########################

import fitz
import subprocess
import os
import time
import win32gui
import win32process
import psutil

PDF_PATH = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_05_09_2025.pdf"
ADOBE_EXE = r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe"
SECTION_TEXT = "Part B - Balance Sheet"


# -------------------- Find page number --------------------
doc = fitz.open(PDF_PATH)
page_number = None

for i, page in enumerate(doc):
    if SECTION_TEXT.lower() in page.get_text().lower():
        page_number = i + 1
        break

doc.close()

if not page_number:
    print("Section not found")
    exit()

print(f"Section found on page {page_number}")


# -------------------- Force-close Acrobat if PDF is open --------------------
def kill_acrobat_if_pdf_open(pdf_path):
    pdf_name = os.path.basename(pdf_path).lower()

    def enum_handler(hwnd, _):
        if win32gui.IsWindowVisible(hwnd):
            title = win32gui.GetWindowText(hwnd).lower()
            if pdf_name in title:
                _, pid = win32process.GetWindowThreadProcessId(hwnd)
                try:
                    p = psutil.Process(pid)
                    if "acrobat" in p.name().lower():
                        print("Force closing Adobe Acrobat (no popup)...")
                        p.kill()   # 🔥 no dialog
                except psutil.NoSuchProcess:
                    pass

    win32gui.EnumWindows(enum_handler, None)


kill_acrobat_if_pdf_open(PDF_PATH)
time.sleep(1)


# -------------------- Open PDF at target page --------------------
subprocess.Popen([
    ADOBE_EXE,
    "/A",
    f"page={page_number}",
    os.path.abspath(PDF_PATH)
])


Section found on page 7


<Popen: returncode: None args: ['C:\\Program Files\\Adobe\\Acrobat DC\\Acrob...>

In [9]:
import winreg
import shlex

def get_default_pdf_opener():
    try:
        # Step 1: Get ProgID for .pdf
        with winreg.OpenKey(winreg.HKEY_CLASSES_ROOT, ".pdf") as key:
            prog_id, _ = winreg.QueryValueEx(key, None)

        # Step 2: Get open command for that ProgID
        cmd_path = rf"{prog_id}\shell\open\command"
        with winreg.OpenKey(winreg.HKEY_CLASSES_ROOT, cmd_path) as key:
            command, _ = winreg.QueryValueEx(key, None)

        # Example command:
        # "C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe" "%1"

        exe = shlex.split(command)[0]
        return exe
    except Exception:
        return None

get_default_pdf_opener()

'C:\\Program Files\\Adobe\\Acrobat DC\\Acrobat\\Acrobat.exe'

In [7]:
import shutil
import os

def find_adobe_executable():
    possible_paths = [
        r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe",
        r"C:\Program Files (x86)\Adobe\Acrobat DC\Acrobat\Acrobat.exe",
        r"C:\Program Files\Adobe\Acrobat Reader DC\Reader\AcroRd32.exe",
        r"C:\Program Files (x86)\Adobe\Acrobat Reader DC\Reader\AcroRd32.exe",
    ]

    for path in possible_paths:
        if os.path.exists(path):
            return path

    # fallback: try PATH
    return shutil.which("Acrobat.exe") or shutil.which("AcroRd32.exe")


In [8]:
find_adobe_executable()

'C:\\Program Files\\Adobe\\Acrobat DC\\Acrobat\\Acrobat.exe'

In [ ]:
######################## Code with checking if adobe is open or not ####################
import fitz
import subprocess
import os
import time
import time
import psutil
import win32gui

def is_adobe_window_open():
    found = False

    def enum_handler(hwnd, _):
        nonlocal found
        if win32gui.IsWindowVisible(hwnd):
            title = win32gui.GetWindowText(hwnd)
            if "Adobe Acrobat" in title:
                found = True

    win32gui.EnumWindows(enum_handler, None)
    return found


def wait_for_adobe(timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        if is_adobe_window_open():
            return True
        time.sleep(0.5)
    return False

def remove_auto_highlights(pdf_path, tag="AUTO_SECTION_HIGHLIGHT"):
    doc = fitz.open(pdf_path)

    for page in doc:
        annots = page.annots()
        if not annots:
            continue

        for annot in annots:
            info = annot.info
            if info and info.get("title") == tag:
                page.delete_annot(annot)

    doc.saveIncr()
    doc.close()


PDF_PATH = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
SECTION_TEXT = "Part B - Balance Sheet"
ADOBE_PATH = r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe"

TAG = "AUTO_SECTION_HIGHLIGHT"

doc = fitz.open(PDF_PATH)

page_number = None
for i, page in enumerate(doc):
    if SECTION_TEXT.lower() in page.get_text().lower():
        page_number = i + 1
        target_page = page
        break

if page_number is None:
    raise Exception("Section not found")

# Add tagged highlight
for rect in target_page.search_for(SECTION_TEXT):
    annot = target_page.add_highlight_annot(rect)
    annot.set_info({"title": TAG})
    annot.update()

doc.saveIncr()
doc.close()


# 🔹 Open at correct page
subprocess.run([
    ADOBE_PATH,
    "/A",
    f"page={page_number}",
    os.path.abspath(PDF_PATH)
])


if wait_for_adobe():
    print("✅ PDF opened in Adobe")
else:
    print("❌ Timeout waiting for Adobe")

remove_auto_highlights(PDF_PATH)

❌ Timeout waiting for Adobe


In [ ]:
######################## Final code with checking if current pdf is open in adobe or not ####################
import fitz  # PyMuPDF
import subprocess
import os
import time
import win32gui

# ================= CONFIG =================

PDF_PATH = r"D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf"
# SECTION_TEXT = "Part B - Balance Sheet"
SECTION_TEXT = "SEGMENT IV"
ADOBE_PATH = r"C:\Program Files\Adobe\Acrobat DC\Acrobat\Acrobat.exe"
TAG = "AUTO_SECTION_HIGHLIGHT"

# ================= WINDOW DETECTION =================

def is_this_pdf_open(pdf_path):
    pdf_name = os.path.basename(pdf_path).lower()
    found = False

    def enum_handler(hwnd, _):
        nonlocal found
        if win32gui.IsWindowVisible(hwnd):
            title = win32gui.GetWindowText(hwnd).lower()
            if pdf_name in title and "adobe acrobat" in title:
                found = True

    win32gui.EnumWindows(enum_handler, None)
    return found


def wait_until_pdf_opens(pdf_path, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        if is_this_pdf_open(pdf_path):
            return True
        time.sleep(0.5)
    return False


def wait_until_pdf_closes(pdf_path, timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        if not is_this_pdf_open(pdf_path):
            return True
        time.sleep(1)
    return False

# ================= HIGHLIGHT LOGIC =================

def add_temp_highlight(pdf_path, text, tag):
    doc = fitz.open(pdf_path)

    page_index = None
    for i, page in enumerate(doc):
        if text.lower() in page.get_text().lower():
            page_index = i
            target_page = page
            break

    if page_index is None:
        doc.close()
        raise Exception("Section text not found")

    for rect in target_page.search_for(text):
        annot = target_page.add_highlight_annot(rect)
        annot.set_info({"title": tag})
        annot.update()

    doc.saveIncr()
    doc.close()

    return page_index + 1  # Adobe uses 1-based indexing


def remove_auto_highlights(pdf_path, tag):
    doc = fitz.open(pdf_path)

    for page in doc:
        annots = page.annots()
        if not annots:
            continue

        for annot in annots:
            info = annot.info
            if info and info.get("title") == tag:
                page.delete_annot(annot)

    doc.saveIncr()
    doc.close()

# ================= MAIN FLOW =================

if __name__ == "__main__":

    # 1️⃣ Add highlight + get page number
    page_number = add_temp_highlight(PDF_PATH, SECTION_TEXT, TAG)
    print(f"✅ Section found and highlighted on page {page_number}")

    # 2️⃣ Open PDF in Adobe
    subprocess.Popen([
        ADOBE_PATH,
        "/A",
        f"page={page_number}",
        os.path.abspath(PDF_PATH)
    ])

    # 3️⃣ Wait until THIS PDF opens
    if not wait_until_pdf_opens(PDF_PATH):
        raise TimeoutError("Adobe did not open the target PDF")

    print("📄 PDF opened in Adobe")

    # 4️⃣ Wait until THIS PDF closes
    print("⏳ Waiting for user to close PDF...")
    wait_until_pdf_closes(PDF_PATH)

    # 5️⃣ Cleanup highlights
    remove_auto_highlights(PDF_PATH, TAG)
    print("🧹 Temporary highlights removed")


✅ Section found and highlighted on page 23
📄 PDF opened in Adobe
⏳ Waiting for user to close PDF...


FzErrorSystem: code=2: cannot open file 'D:\Nexensus_Projects\pdfFoms\pdfs\AOC-4 NBFC_Form AOC4 NBFCIND AS_30_08_2025.pdf': Permission denied

In [4]:
remove_auto_highlights(PDF_PATH, TAG)